# CPG-RL **v2.4** 訓練：智元 D1 Max · MJX · Colab GPU（2026-09-08）

基準 `A_kp250_walk`（LS / kp250 / abad60 / kd2 / wheel_kd 0.5，實機 trip17 兩趟零中止走完）。
14 維動作（每腿 mux/muy/ω ＋ body sway x,y）、70 維 obs，隨機化與護欄依
`task7/docs/H_實機obs盤點_2026-09-08.md`；設計 `docs/superpowers/specs/2026-09-08-cpg-rl-v2-retrain-design.md`。

**env 住在 repo（`task7/inference/rl_env_max.py`），本 notebook 只有：安裝 → clone → 校準 → 訓練 → 存檔。**
reward preset：**`v2.4`**（權重定義與校準依據都在 `rl_env_max.PRESETS`；`diag/rl_calibrate.py` 量的）。
改 env 請改 repo、push、重新 clone（第 3 格會印 clone 到的 commit），不要在這裡貼程式。

流程：全部執行 → 第 5 格看「基準校準」（基準動作不該 done、不該碰護欄）→ 第 6 格第一個 eval
看實測步率（60M 超過 3 小時就砍 `TIMESTEPS` 到 40M 重跑）→ 下載 `cpg_rl_max_v2_4_params.pkl`
→ 本機 `local_infer_max.py --params … --secs 60 --perturb 12 --compare --video` 判 G3–G7。


In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

# 版本鎖死，不要放寬。brax 的預設 activation/分布由版本決定，而 activation
# 不匹配時 brax 載權重【不會報錯】，只會讓 policy 靜默錯亂。
# 這裡的版本 = 本機推論端 (task7/inference/) 的版本。
#
# jax<0.10：brax 0.14.2 的 ppo.train (train.py:756) 用 jax.device_put_replicated，
# 該 API 在 jax 0.10 已被移除（本機 jax 0.10.2 實測直接 AttributeError）。
# 用 jax[cuda12] 這個 extra 是為了讓 jaxlib 與 CUDA plugin 一起降到相容版本，
# 只寫 "jax<0.10" 會留下版本不合的 jaxlib/cuda plugin。
# ⚠️ 這一項無法在本機驗證：裝完務必看下一格印出的 devices 有沒有 cuda。
#    掉回 CPU 或裝不起來 → 回報，【不要】自行改 brax 版本（會動到 activation 預設值）。
!pip install -q "brax==0.14.2" "mujoco==3.10.0" "mujoco-mjx==3.10.0" "jax[cuda12]<0.10" mediapy
print("done")

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())   # 要看到 cuda

# 版本斷言：不匹配當場停住，不要繞過。訓練跑完才發現行為對不上就白費了。
import brax, mujoco
print("brax", brax.__version__, "| mujoco", mujoco.__version__)
assert brax.__version__ == "0.14.2", (
    f"brax 版本為 {brax.__version__}，本機推論端是 0.14.2。"
    "版本不同會改變 make_ppo_networks 的預設 activation，"
    "而 activation 不匹配時 brax 載權重【不會報錯】，只會讓 policy 行為錯亂。"
    "請回到上一格重跑安裝（Colab 有時需要「執行階段 → 重新啟動工作階段」才會生效）。"
)
assert mujoco.__version__ == "3.10.0", (
    f"mujoco 版本為 {mujoco.__version__}，本機是 3.10.0。"
    "MJX 的接觸/求解器行為隨版本改變，訓練與推論不同版會讓步態對不上。"
)
# jax 0.10 移除了 device_put_replicated，而 brax 0.14.2 的 ppo.train 會用它。
# 在這裡早死，不要拖到訓練那一格編譯完才炸。
assert hasattr(jax, "device_put_replicated"), (
    f"jax {jax.__version__} 已移除 device_put_replicated，brax 0.14.2 的 ppo.train 會失敗。"
    "需要 jax<0.10。若 Colab 無法在此版本下取得 GPU 支援，請回報——"
    "換 brax 版本會改變 make_ppo_networks 的預設 activation，那會讓權重與本機推論端靜默不匹配。"
)
if not any(d.platform == "gpu" for d in jax.devices()):
    print("⚠️ 沒抓到 GPU。確認「執行階段 → 變更類型 → GPU」，"
          "以及上一格的 jax[cuda12]<0.10 是否把 CUDA 支援裝掉了。")
print("版本 OK")

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/HGLLLLL/RBTDOG_SIM.git"
BRANCH = "main"
DEST = "rbtdog_sim"          # repo 名是大寫 RBTDOG_SIM，明寫目的地

if not os.path.exists(DEST):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DEST], check=True)
sys.path.insert(0, f"{DEST}/task7/inference")
print("clone 到的 commit：",
      subprocess.run(["git", "-C", DEST, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
# ★ 訓練模型 zgws_mjx_kp250.xml 零 STL 相依，不需要 fetch_assets.sh


In [ ]:
import jax, jax.numpy as jnp, numpy as np, mujoco
import rl_env_max as re
import obs_max, gait_baseline as gb, max_model as mm

PRESET = "v2.4"
W = re.weights_of(PRESET)
print("preset", PRESET, "權重", W)
LAYOUT = W["ACT_LAYOUT"]
ACT_DIM = re.LAYOUT_DIMS[LAYOUT]
HEAD = bool(W.get("HEAD_OBS", False))
OBS_DIM = obs_max.obs_dim(ACT_DIM, HEAD)
print("layout", LAYOUT, "act", ACT_DIM, "obs", OBS_DIM, "head_err in obs", HEAD, "scene", mm.SCENE_MJX_KP250)
print("基準", gb.BASELINE_A)
print("護欄 TAU_BAR", re.TAU_BAR, "ERR_BAR", re.ERR_BAR, "| sway ±", re.SWAY_MAX, "斜率", re.SWAY_SLEW)
assert (ACT_DIM, OBS_DIM) in ((14, 70), (10, 66), (10, 67))
assert list(gb.BASELINE_A["kp3"]) == [60.0, 250.0, 250.0], "ABAD 必須是 60"
if LAYOUT == "nomux":
    print("★ mu_x 固定 =", gb.BASELINE_A["mu_x"], "（不在動作空間），速度只靠 ω；指令範圍", W["CMD_VX"])


In [ ]:
# ---- ★ 基準校準：A 步態是固定動作（sway=0），每一項 reward 在它身上值多少？----
# 兩個目的：(1) env 有沒有重現開迴路基準（本機 G0：speed 0.34、roll_pk 3.9°、exec 前 0.88/後 1.47）
#          (2) 護欄在基準上不該被碰到（tau_pk 平均要 < TAU_BAR、err_pk < ERR_BAR）
env = re.MaxCpgEnv(preset=PRESET)
assert env.preset == PRESET and env.w == W
assert (env.action_size, env.observation_size) == (ACT_DIM, OBS_DIM)
assert env.sys.actuator_biastype[0] == mujoco.mjtBias.mjBIAS_AFFINE
kp_xml = np.asarray(env.sys.actuator_gainprm[np.asarray(mm.LEG_ACT_IDX), 0])
assert np.allclose(kp_xml, np.tile(mm.KP3_A, 4)), f"訓練模型增益 {kp_xml[:3]} ≠ KP3_A"
jit_reset, jit_step = jax.jit(env.reset), jax.jit(env.step)
A_BASE = jnp.array(re.baseline_action(LAYOUT))
CAL_VX = 0.30 if W["CMD_VX"][1] > 0.36 else 0.15                # 校準用固定指令（同 diag/rl_calibrate.py）
s = jit_reset(jax.random.PRNGKey(0))
s = s.replace(info={**s.info, "cmd": jnp.array([CAL_VX, 0.0])})
print("reset ok, obs", s.obs.shape, "height %.4f m" % float(s.pipeline_state.qpos[2]))

import time as _t
_t0 = _t.time()
acc = {k: [] for k in re.METRIC_KEYS}
xs = []
for i in range(500):                      # 10 s
    s = jit_step(s, A_BASE)
    assert float(s.done) == 0.0, f"基準動作在第 {i} 步 done —— env 有問題，不要訓"
    if i >= 250:
        for k in re.METRIC_KEYS:
            acc[k].append(float(s.metrics[k]))
        xs.append(float(s.pipeline_state.qpos[0]))
print(f"[基準 10 s，後半段平均]  " + "  ".join(f"{k} {np.mean(v):.3f}" for k, v in acc.items())
      + f"   ({_t.time() - _t0:.0f}s)")
print(f"[護欄] tau_pk 平均 {np.mean(acc['tau_pk']):.1f} / 最大 {np.max(acc['tau_pk']):.1f}（TAU_BAR {re.TAU_BAR}）"
      f"   err_pk 平均 {np.mean(acc['err_pk']):.3f} / 最大 {np.max(acc['err_pk']):.3f}（ERR_BAR {re.ERR_BAR}）")
print("       基準步態不該常態碰到護欄 —— 若 tau_pk 平均 > TAU_BAR 或 err_pk 平均 > ERR_BAR，先回本機查再訓")
print(f"[對照] 本機 local_infer_max --dummy：speed_travel 0.34 m/s、roll_pk 3.86°、exec 前 0.88 / 後 1.47")

# ---- ★ reward 佔比檢查（權重是量出來校準的，這裡再驗一次；與本機 diag/rl_calibrate.py 同算法）----
pos = np.mean(acc["t_pos"])
share = {k: np.mean(acc[k]) / pos for k in re.TERM_KEYS if k != "t_pos"}
print("[佔比] " + "  ".join(f"{k[2:]} {100*v:.1f}%" for k, v in share.items() if v > 0.001))
print("[對照] 本機 rl_calibrate  v2.1：yaw 28% exec 34% roll+rate 14% pitch+rate 5% vx 13.5%"
      "  |  v2.2：vx 37% yaw 24% exec 19% sym 12% roll+rate 7.7% pitch+rate 4.3%")
if W["EXEC_MODE"] == "rate":
    print(f"[執行率] 基準 前 {np.mean(acc['exec_f']):.2f} / 後 {np.mean(acc['exec_r']):.2f}（本機 Trace：0.88 / 1.47，應同量級）")
roll_sh = share["t_roll"] + share["t_rollrate"]
pitch_sh = share["t_pitch"] + share["t_pitchrate"]
if PRESET in re.CAL_BANDS:
    b = re.CAL_BANDS[PRESET]
    assert b["roll"][0] <= roll_sh <= b["roll"][1], f"roll 兩項佔正項 {roll_sh:.1%}，不在 {b['roll']}：權重跟本機校準對不上，不要訓"
    assert b["pitch"][0] <= pitch_sh <= b["pitch"][1], f"pitch 兩項佔 {pitch_sh:.1%}，不在 {b['pitch']}"
    assert share["t_exec"] <= b["exec_max"] and roll_sh > pitch_sh
assert share["t_taubar"] < 0.01 and share["t_errbar"] < 0.01, "基準動作碰到護欄 —— env 或模型有問題，不要訓"
print(f"[佔比] roll+rate {roll_sh:.1%}  pitch+rate {pitch_sh:.1%}  exec {share['t_exec']:.1%}  ✅ 與本機校準一致")


In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

env = re.MaxCpgEnv(preset=PRESET)
# ⚠️ 網路尺寸與 normalize_observations=True 必須與本機推論端 (local_infer_max.py) 逐項相同。
network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=(256, 256, 128),
    value_hidden_layer_sizes=(256, 256, 256))

TIMESTEPS = 60_000_000
train_fn = functools.partial(
    ppo.train, num_timesteps=TIMESTEPS, num_evals=20, episode_length=1000,
    num_envs=2048, batch_size=256, num_minibatches=32, unroll_length=20,
    num_updates_per_batch=4, learning_rate=3e-4, entropy_cost=1e-2,
    discounting=0.97, normalize_observations=True,
    network_factory=network_factory, randomization_fn=re.domain_randomize, seed=0)

_t0 = time.time()
rewards = []


def progress(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0))
    rewards.append((step, r))
    L = float(metrics.get("eval/avg_episode_length", 1.0)) or 1.0

    def ps(k):
        return float(metrics.get(f"eval/episode_{k}", 0.0)) / L

    el = time.time() - _t0
    rate = step / max(el, 1e-9)
    print(f"step {step:>11,} R {r:7.2f} | roll {ps('roll'):4.2f}° pitch {ps('pitch'):4.2f}° "
          f"exec f/r {ps('exec_f'):.2f}/{ps('exec_r'):.2f} yawerr {ps('yawerr'):.3f} "
          f"vxerr {ps('vxerr'):.3f} | tau_pk {ps('tau_pk'):5.1f} err_pk {ps('err_pk'):.3f} "
          f"sway {ps('sway_x'):.0f}/{ps('sway_y'):.0f}mm len {L:.0f} | "
          f"{el:.0f}s {rate / 1e3:.0f}k步/s → {TIMESTEPS / 1e6:.0f}M 約 {TIMESTEPS / max(rate, 1) / 60:.0f} 分")


# 指標怎麼讀（每控制步平均）：roll/pitch 越小越好（基準動作在 env 裡 roll 2.35° / pitch 1.41°）；exec f/r 越接近 1 越好；
# tau_pk 要一路 < 58、err_pk < 0.45（護欄）；sway 是 policy 實際用的質心位移；len 1000 = 沒提早 done。
make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)
print("training done —— preset", PRESET)


In [ ]:
import matplotlib.pyplot as plt
plt.plot([s for s, _ in rewards], [r for _, r in rewards], marker="o")
plt.xlabel("env steps"); plt.ylabel("eval reward"); plt.grid(True); plt.show()


In [ ]:
from brax.io import model
model.save_params("cpg_rl_max_v2_4_params.pkl", params)
print("已存 cpg_rl_max_v2_4_params.pkl → 下載放 task7/weights/，本機（rbtdog 環境）跑：")
print("  conda run --no-capture-output -n rbtdog python task7/inference/local_infer_max.py "
      "--params task7/weights/cpg_rl_max_v2_4_params.pkl --preset v2.4 --secs 60 --perturb 12 --compare --video")
print("★ 驗收在原始網格模型上做；G7（力矩×1.2<70、誤差×1.14<0.6）任一不過就不上機。")
